# Studio OS — Wan 2.2 Colab Launcher
Select a GPU runtime (Runtime → Change runtime type), then Run all.
Picks the Wan 2.2 variant that fits the GPU, clones only `studio-os`, downloads weights, and generates one test video.


In [ ]:
!nvidia-smi
import subprocess, torch
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f"GPU VRAM: {vram_gb:.1f} GB")
if vram_gb >= 70:
    TASK, REPO = "t2v-A14B", "Wan-AI/Wan2.2-T2V-A14B"
elif vram_gb >= 20:
    TASK, REPO = "ti2v-5B", "Wan-AI/Wan2.2-TI2V-5B"
else:
    raise SystemExit("No suitable GPU (need at least 24GB). Change runtime to a GPU with more memory.")
print("Selected:", TASK, REPO)


In [ ]:
!git clone https://github.com/rayeshiacross-collab/studio-os.git /content/studio-os


In [ ]:
!git clone https://github.com/Wan-Video/Wan2.2.git /content/Wan2.2
%cd /content/Wan2.2
!pip install -q -r requirements.txt
!pip install -q "huggingface_hub[cli]"


In [ ]:
!huggingface-cli download {REPO} --local-dir /content/ckpt/{TASK}


In [ ]:
PROMPT = "A calm ocean at sunset, gentle waves, cinematic lighting"
SIZE = "1280*704" if TASK == "ti2v-5B" else "1280*720"
!python generate.py --task {TASK} --size {SIZE} --ckpt_dir /content/ckpt/{TASK} --offload_model True --convert_model_dtype --prompt "{PROMPT}" --save_file /content/wan22_test.mp4


In [ ]:
from IPython.display import Video
Video("/content/wan22_test.mp4", embed=True)
